Este bloco cria o Recomendador de livros

In [2]:
import sqlite3
import pandas as pd 
from fuzzywuzzy import process #facilita o reconhecimento de padrões de linguagem natural (erros de digitação/similaridade)
import textwrap #melhoria na visualização do resultado ('overflow')

#Conecta ao banco e carrega os dados da query pra variável ser usada depois
conn = sqlite3.connect("biblioteca.db")
df_livros = pd.read_sql_query("""
    SELECT l.id, l.title, l.authors, c.name AS categoria, 
           AVG(d.nota) AS media_avaliacao, COUNT(d.id) AS total_downloads
    FROM livros l
    JOIN categorias c ON l.categoria_id = c.id
    LEFT JOIN downloads d ON l.id = d.livro_id
    GROUP BY l.id
""", conn)
conn.close()

#Mostra as escolhas possíveis para o usuário
print("Escolha o critério de pesquisa:")
print("1 - Título")
print("2 - Autor")
print("3 - Categoria")

# Função que recomenda os livros
def recomendar_livros(entrada, criterio):
    
    entrada = str(entrada).strip().lower()
        # Faz um tratamento de correção ortográfica e transforma a entrada em minúsculas
    if not entrada:
        return exibir_recomendacoes(
            "Nenhum termo foi inserido.\n\nAbaixo deixamos como recomendação nossos livros mais populares:\n",
            df_livros.sort_values(by=["media_avaliacao", "total_downloads"], ascending=False).head(5)
        )
    if criterio == 1:  # Pesquisa por Título
    # Encontra a melhor correspondência para o título inserido 
        similaridade = process.extractOne(entrada, df_livros['title'].str.lower(), score_cutoff=60)
        if similaridade:
                # Seleciona o livro que corresponde ao título encontrado
                livro = df_livros[df_livros['title'].str.lower() == similaridade[0]].iloc[0]
                # Pega a categoria do livro encontrado
                categoria = livro['categoria']
                # Busca recomendações de livros na mesma categoria
                recomendacoes = df_livros[df_livros['categoria'] == categoria].head(5)
        
                # Verifica se a similaridade é maior que 80%
                if similaridade[1] > 80:
                    return exibir_recomendacoes(
                    f"O livro '{similaridade[0].title()}' foi encontrado.\nConfira abaixo as recomendações de livros da mesma categoria:\n", 
                    recomendacoes
                    )
                else:
                    return exibir_recomendacoes(
                        f"Aqui estão os resultados da busca '{entrada}' por Título:\n", 
                    recomendacoes
                     )
    
    if criterio == 2:  #Pesquisa por Autor
        similaridade = process.extractOne(entrada, df_livros['authors'].str.lower(), score_cutoff=60)
        if similaridade:
            if similaridade[1] >= 80:  #Formata a resposta diferente se a similaridade for maior ou igual a 80%
                return exibir_recomendacoes(f"Autor '{similaridade[0].title()}' encontrado.\nConfira abaixo as recomendações de obras desse autor com as melhores avaliações:\n", 
                                            df_livros[df_livros['authors'].str.lower().str.contains(similaridade[0], case=False, na=False)].head(5))
            else: #Responde de forma generalizada sobre o termo pesquisado
                return exibir_recomendacoes(f"Aqui estão os resultados da busca '{entrada}' por Autor:\n", 
                                            df_livros[df_livros['authors'].str.lower().str.contains(similaridade[0], case=False, na=False)].head(5))
    
    if criterio == 3:  #Pesquisa por Categoria
        similaridade = process.extractOne(entrada, df_livros['categoria'].str.lower(), score_cutoff=60)
        if similaridade:
            recomendacoes = df_livros[df_livros['categoria'].str.lower() == similaridade[0]].head(5)
            return exibir_recomendacoes(f"Aqui estão os resultados da busca '{entrada}' por Categoria:\n", recomendacoes)
    
    recomendacoes = df_livros.sort_values(by=["media_avaliacao", "total_downloads"], ascending=False).head(5)
    return exibir_recomendacoes(f"Nenhum resultado encontrado para '{entrada}'.\n\nAbaixo estão as recomendações dos livros mais populares em nossa biblioteca:\n", recomendacoes)

#Tratamentos das recomendações:
#Ordena os livros encontrados por ranking e remove os nomes duplicados (se houver)
df_livros = df_livros.sort_values(by=["media_avaliacao"], ascending=False).drop_duplicates(subset=["title"], keep="first")

#Remove livros com avaliação inferior a 2.59
df_livros = df_livros[df_livros["media_avaliacao"] >= 2.59]

#Formata a resposta
def exibir_recomendacoes(mensagem, recomendacoes):
    recomendacoes["media_avaliacao"] = recomendacoes["media_avaliacao"].map(lambda x: round(x, 2) if pd.notna(x) else 0.0)
    relevancias = recomendacoes["media_avaliacao"].astype(float).tolist()
    
    return {
        "mensagem": mensagem,
        "recomendacoes": recomendacoes[['title', 'authors', 'categoria', 'media_avaliacao']].rename(
            columns={"title": "Título", "authors": "Autor", "categoria": "Categoria", "media_avaliacao": "Avaliação"}
        ).map(lambda x: x.title() if isinstance(x, str) else x)
    }

criterio_usuario = int(input("Digite o número correspondente: ").strip())
entrada_usuario = input("Digite o termo de pesquisa: ")
resultado = recomendar_livros(entrada_usuario, criterio_usuario)

#Mostra o retorno da busca e as recomendações 
print("\n" + resultado["mensagem"])
if resultado["recomendacoes"] is not None:
    print(f"{'Título'.ljust(30)} {'Autor'.ljust(25)} {'Categoria'.ljust(20)} {'Avaliação'.ljust(30)}")
    for _, row in resultado["recomendacoes"].iterrows():
        titulo = textwrap.shorten(str(row["Título"]), width=30, placeholder="...").ljust(30)
        autor = textwrap.shorten(str(row["Autor"]), width=25, placeholder="...").ljust(25)
        categoria = textwrap.shorten(str(row["Categoria"]), width=20, placeholder="...").ljust(20)
        avaliacao = textwrap.shorten(str(row["Avaliação"]),width=30, placeholder="...").ljust(30)
        print(f"{titulo} {autor} {categoria} {avaliacao}")

Escolha o critério de pesquisa:
1 - Título
2 - Autor
3 - Categoria

Aqui estão os resultados da busca 'oz' por Título:

Título                         Autor                     Categoria            Avaliação                     
The Wonderful Wizard Of Oz     L. Frank Baum             Juvenile Fiction     3.35                          
Treasure Island                Robert Louis Stevenson    Juvenile Fiction     3.19                          
Little Women                   Louisa May Alcott         Juvenile Fiction     3.12                          
Emma                           Jane Austen               Juvenile Fiction     3.12                          
Wuthering Heights              Emily Brontë             Juvenile Fiction     3.0                           
